# Лекция: Вероятностные распределения в Python

**Дисциплина:** Введение в анализ больших данных  
**Задание 2** (адаптация с языка R на Python)

В оригинальном задании на R рассматриваются:
- Нормальное распределение
- Равномерное распределение
- Распределение Стьюдента (t-распределение)

и построение гистограмм с наложением кривой плотности.

В Python для работы с распределениями используются библиотеки:
- **NumPy** — генерация случайных чисел
- **SciPy** (`scipy.stats`) — теоретические распределения, плотности, функции распределения
- **Matplotlib** / **Seaborn** — визуализация

В этой тетради мы разберём основные классы и методы, необходимые для выполнения Задания 2, и решим все примеры.


## 0. Импорт библиотек


In [ ]:
# !pip install numpy scipy matplotlib seaborn pandas

import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# Настройки графиков
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style("whitegrid")

print("NumPy:", np.__version__)
print("SciPy:", stats.__version__ if hasattr(stats, '__version__') else "ok")


---
## Краткая теория

**Распределение числовой случайной величины** — функция, которая однозначно определяет вероятность того, что случайная величина принимает заданное значение или принадлежит заданному интервалу.

**Закон распределения** — соотношение между возможными значениями случайной величины и соответствующими им вероятностями.

### Основные распределения в задании

| Распределение | Параметры | В R | В Python (scipy.stats) |
|---------------|-----------|-----|------------------------|
| Нормальное | mean (μ), sd (σ) | `rnorm`, `dnorm` | `stats.norm` |
| Стандартное нормальное | μ=0, σ=1 | `rnorm(n)` | `stats.norm(0,1)` |
| Равномерное | a, b | `runif` | `stats.uniform` |
| Стьюдента (t) | df (степени свободы) | `rt`, `dt` | `stats.t` |

### Основные методы объектов распределений SciPy
- `.rvs(size=n)` — генерация случайной выборки (Random Variates)
- `.pdf(x)` — плотность вероятности (Probability Density Function)
- `.cdf(x)` — функция распределения (Cumulative Distribution Function)
- `.ppf(q)` — квантиль (обратная к cdf)
- `.mean()`, `.std()`, `.var()` — теоретические моменты


---
## 1. Нормальное распределение

Нормальное распределение — одно из самых важных в статистике.  
Характеризуется симметричной колоколообразной формой.

**Плотность вероятности:**

$$
f(x) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left( -\frac{(x-\mu)^2}{2\sigma^2} \right)
$$

**Стандартное нормальное распределение:** μ = 0, σ = 1.


In [ ]:
# Пример 1. Случайная выборка из стандартного нормального распределения, n=10
# В R: rnorm(n=10)

np.random.seed(42)  # для воспроизводимости

x1 = stats.norm.rvs(loc=0, scale=1, size=10)
# или проще:
# x1 = np.random.normal(0, 1, 10)
# x1 = stats.norm(0, 1).rvs(10)

print("Выборка x1 (стандартное нормальное, n=10):")
print(x1)
print(f"\nВыборочное среднее: {x1.mean():.4f}")
print(f"Выборочное ст. отклонение: {x1.std(ddof=1):.4f}")


In [ ]:
# Пример 2. Два вектора длины n=100
# а) mean=1, sd=3
# б) mean=-1, sd=2

n = 100
np.random.seed(123)

# а)
x_a = stats.norm.rvs(loc=1, scale=3, size=n)
print("Вектор а) mean=1, sd=3 (первые 10 значений):")
print(x_a[:10])
print(f"Среднее: {x_a.mean():.3f}, Ст. откл.: {x_a.std(ddof=1):.3f}\n")

# б)
x_b = stats.norm.rvs(loc=-1, scale=2, size=n)
print("Вектор б) mean=-1, sd=2 (первые 10 значений):")
print(x_b[:10])
print(f"Среднее: {x_b.mean():.3f}, Ст. откл.: {x_b.std(ddof=1):.3f}")


### Гистограмма + кривая плотности для нормального распределения


In [ ]:
# Гистограмма для выборки из нормального распределения (mean=15, sd=5), n=50
# Аналог примера из задания

np.random.seed(7)
X = stats.norm.rvs(loc=15, scale=5, size=50)

fig, ax = plt.subplots(figsize=(10, 6))

# Гистограмма (density=True → нормированная, как freq=FALSE в R)
ax.hist(X, bins=20, density=True, color="lightblue", edgecolor="black",
        alpha=0.7, label="Гистограмма")

# Кривая плотности (теоретическая)
x_grid = np.linspace(X.min() - 2, X.max() + 2, 300)
pdf = stats.norm.pdf(x_grid, loc=15, scale=5)
ax.plot(x_grid, pdf, color="red", linewidth=2, label="Теоретическая плотность N(15, 5)")

ax.set_xlabel("Переменная X")
ax.set_ylabel("Плотность вероятности")
ax.set_title("Гистограмма, совмещенная с кривой плотности\nНормальное распределение N(15, 5)")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Гистограммы для выборок из Примера 2

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# а) N(1, 3)
axes[0].hist(x_a, bins=20, density=True, color="skyblue", edgecolor="black", alpha=0.7)
x_grid_a = np.linspace(x_a.min()-1, x_a.max()+1, 300)
axes[0].plot(x_grid_a, stats.norm.pdf(x_grid_a, 1, 3), color="red", lw=2)
axes[0].set_title("N(mean=1, sd=3), n=100")
axes[0].set_xlabel("x")
axes[0].set_ylabel("Плотность")

# б) N(-1, 2)
axes[1].hist(x_b, bins=20, density=True, color="lightgreen", edgecolor="black", alpha=0.7)
x_grid_b = np.linspace(x_b.min()-1, x_b.max()+1, 300)
axes[1].plot(x_grid_b, stats.norm.pdf(x_grid_b, -1, 2), color="darkred", lw=2)
axes[1].set_title("N(mean=-1, sd=2), n=100")
axes[1].set_xlabel("x")
axes[1].set_ylabel("Плотность")

plt.suptitle("Нормальные распределения (Пример 2)", fontsize=14)
plt.tight_layout()
plt.show()


---
## 2. Равномерное распределение

Равномерное распределение на отрезке [a, b] — все значения внутри интервала равновероятны.

**Плотность:**  
$$
f(x) = \frac{1}{b-a}, \quad a \le x \le b
$$

В R: `runif(n, min=0, max=1)`  
В Python: `stats.uniform(loc=a, scale=b-a)`


In [ ]:
# Пример 3. Случайная выборка из равномерного распределения на [0; 1], n=20
# В R: runif(n=20)

np.random.seed(99)
u = stats.uniform.rvs(loc=0, scale=1, size=20)
# или: u = np.random.uniform(0, 1, 20)

print("Выборка u ~ Uniform[0, 1], n=20:")
print(u)
print(f"\nМинимум: {u.min():.4f}, Максимум: {u.max():.4f}")
print(f"Среднее: {u.mean():.4f} (теоретическое = 0.5)")


In [ ]:
# Гистограмма + плотность для равномерного распределения

np.random.seed(99)
u_large = stats.uniform.rvs(loc=0, scale=1, size=500)  # больше точек для наглядности

fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(u_large, bins=20, density=True, color="orange", edgecolor="black",
        alpha=0.7, label="Гистограмма")

# Теоретическая плотность (горизонтальная линия на высоте 1)
x_grid = np.linspace(-0.1, 1.1, 300)
pdf = stats.uniform.pdf(x_grid, loc=0, scale=1)
ax.plot(x_grid, pdf, color="blue", linewidth=2.5, label="Теоретическая плотность Uniform[0,1]")

ax.set_xlabel("u")
ax.set_ylabel("Плотность вероятности")
ax.set_title("Гистограмма + кривая плотности\nРавномерное распределение на [0; 1]")
ax.legend()
ax.set_ylim(0, 1.5)
plt.tight_layout()
plt.show()


---
## 3. Распределение Стьюдента (t-распределение)

Распределение Стьюдента возникает при оценке среднего нормальной выборки, когда дисперсия неизвестна.

Параметр: **df** — число степеней свободы.

- При df → ∞ t-распределение стремится к стандартному нормальному.
- При малых df (особенно df=1) имеет тяжёлые хвосты.

В R: `rt(n, df)`, `dt(x, df)`  
В Python: `stats.t(df)`


In [ ]:
# Пример 4. Плотности распределения Стьюдента с df=1 и df=5
# В задании указано: x = seq(-5, 5, by=0.01)

x = np.arange(-5, 5.01, 0.01)

# Плотности
pdf_df1 = stats.t.pdf(x, df=1)
pdf_df5 = stats.t.pdf(x, df=5)
pdf_norm = stats.norm.pdf(x, 0, 1)  # для сравнения

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(x, pdf_df1, color="red", linewidth=2, label="t(df=1) — распределение Коши")
ax.plot(x, pdf_df5, color="blue", linewidth=2, label="t(df=5)")
ax.plot(x, pdf_norm, color="black", linewidth=2, linestyle="--", label="N(0,1) — стандартное нормальное")

ax.set_xlabel("x")
ax.set_ylabel("Плотность вероятности")
ax.set_title("Плотности распределения Стьюдента (df=1 и df=5)\nсравнение со стандартным нормальным")
ax.legend()
ax.set_xlim(-5, 5)
plt.tight_layout()
plt.show()


In [ ]:
# Генерация выборок из t-распределения и гистограммы

np.random.seed(42)
sample_t1 = stats.t.rvs(df=1, size=500)
sample_t5 = stats.t.rvs(df=5, size=500)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# df=1
axes[0].hist(sample_t1, bins=40, density=True, color="salmon", edgecolor="black", alpha=0.7)
x_grid = np.linspace(-10, 10, 400)
axes[0].plot(x_grid, stats.t.pdf(x_grid, df=1), color="darkred", lw=2)
axes[0].set_title("Выборка из t(df=1), n=500\n+ теоретическая плотность")
axes[0].set_xlabel("x")
axes[0].set_ylabel("Плотность")
axes[0].set_xlim(-10, 10)

# df=5
axes[1].hist(sample_t5, bins=30, density=True, color="lightblue", edgecolor="black", alpha=0.7)
axes[1].plot(x_grid, stats.t.pdf(x_grid, df=5), color="darkblue", lw=2)
axes[1].set_title("Выборка из t(df=5), n=500\n+ теоретическая плотность")
axes[1].set_xlabel("x")
axes[1].set_ylabel("Плотность")
axes[1].set_xlim(-6, 6)

plt.suptitle("Распределение Стьюдента (Пример 4)", fontsize=14)
plt.tight_layout()
plt.show()


---
## 4. Полезные дополнительные примеры

### Сравнение нескольких нормальных распределений


In [ ]:
x = np.linspace(-10, 15, 500)

plt.figure(figsize=(10, 6))
plt.plot(x, stats.norm.pdf(x, 0, 1), label="N(0, 1)", lw=2)
plt.plot(x, stats.norm.pdf(x, 1, 3), label="N(1, 3)", lw=2)
plt.plot(x, stats.norm.pdf(x, -1, 2), label="N(-1, 2)", lw=2)
plt.plot(x, stats.norm.pdf(x, 15, 5), label="N(15, 5)", lw=2)

plt.title("Сравнение нормальных распределений с разными параметрами")
plt.xlabel("x")
plt.ylabel("Плотность")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Генерация и описание выборки одной функцией


In [ ]:
def describe_sample(sample, name="Выборка"):
    """Краткое описание числовой выборки"""
    print(f"=== {name} ===")
    print(f"n = {len(sample)}")
    print(f"Минимум      : {sample.min():.4f}")
    print(f"Максимум     : {sample.max():.4f}")
    print(f"Среднее      : {sample.mean():.4f}")
    print(f"Медиана      : {np.median(sample):.4f}")
    print(f"Ст. отклонение: {sample.std(ddof=1):.4f}")
    print(f"Дисперсия    : {sample.var(ddof=1):.4f}")
    print()

describe_sample(x1, "Стандартное нормальное (n=10)")
describe_sample(x_a, "N(1, 3)")
describe_sample(u, "Uniform[0,1] (n=20)")


---
## Шпаргалка: R → Python (распределения)

| Задача в R | Python (scipy.stats / numpy) |
|------------|------------------------------|
| `rnorm(n, mean, sd)` | `stats.norm.rvs(loc=mean, scale=sd, size=n)` |
| `dnorm(x, mean, sd)` | `stats.norm.pdf(x, loc=mean, scale=sd)` |
| `pnorm(x, mean, sd)` | `stats.norm.cdf(x, loc=mean, scale=sd)` |
| `qnorm(p, mean, sd)` | `stats.norm.ppf(p, loc=mean, scale=sd)` |
| `runif(n, min, max)` | `stats.uniform.rvs(loc=min, scale=max-min, size=n)` |
| `dunif(x, min, max)` | `stats.uniform.pdf(x, loc=min, scale=max-min)` |
| `rt(n, df)` | `stats.t.rvs(df=df, size=n)` |
| `dt(x, df)` | `stats.t.pdf(x, df=df)` |
| `hist(..., freq=FALSE)` + `lines(density())` | `plt.hist(..., density=True)` + `plt.plot(x, pdf)` |
| `set.seed(42)` | `np.random.seed(42)` |

---
## Рекомендации

1. Документация SciPy: [scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html)
2. Для красивых графиков часто используют `seaborn.histplot(..., kde=True)`
3. Всегда фиксируйте зерно генератора (`np.random.seed`) для воспроизводимости результатов.

**Удачи с выполнением Задания 2!**
